# Lesson 13 - Agent Memory wit Cognee Knowledge Graphs


## Setup

Dis notebook dey show how to build one sharp **coding assistant** wey get persistent memory using [**Cognee**](https://www.cognee.ai/) knowledge graphs and the **Microsoft Agent Framework** (MAF).

Cognee dey change unstructured text to structured, queryable knowledge graph wey backend na vector embeddings — dis one go give your agent beta, relationship-aware long-term memory.

### Wetin You Go Learn
1. **Build Knowledge Graphs**: Change developer profiles and better practice dem to structured, queryable knowledge.
2. **Integrate Cognee with MAF**: Use `@tool` functions make MAF agent fit query Cognee knowledge graph.
3. **Session-Aware Conversations**: Hold context well well across many questions for the same session.
4. **Long-Term Memory**: Keep important knowledge across sessions and fit carry am come back for new talks.

### Wetin You Need Before
- Python 3.9+
- Redis wey dey run for local (`docker run -d -p 6379:6379 redis`) for session management
- LLM API key (like OpenAI) — set `LLM_API_KEY` for `.env`
- `CACHING=true` for `.env` (dis one na must for Cognee sessions)
- Microsoft Foundry project wey get deployed chat model
- `AZURE_AI_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT_NAME` for `.env`
- Azure CLI authenticated (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Kain Memory Wey Agent Get

Dis notebook dey explore di same three kain memory from di main Lesson 13 notebook, but e dey use Cognee as di long-term memory backend:

| Memory Type | Mechanism | Lifetime |
|---|---|---|
| **Working** | `agent.create_session()` (MAF) | One conversation |
| **Short-term** | Cognee session cache (Redis) | One session |
| **Long-term** | Cognee knowledge graph + vectors | Permanent |

### Cognee Memory Architecture
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Ready Cognee Storage


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Part 1 — Di Knowledge Base We Dey Build

We dey gather three kain data to create one correct knowledge base for our coding assistant:

1. **Developer Profile** — personal skill and technical background
2. **Python Best Practices** — di Zen of Python with correct guidelines
3. **Historical Conversations** — old Q&A wey dey happen between developers and AI assistants


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualize di Knowledge Graph

Cognee fit show una interactive HTML visualization of di entities dem and relationships we e take comot.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Make Brain Dey Sharp Wit Memify

`memify()` dey check knowledge graph come create smart rules — e dey find patterns, beta ways to do things, plus how concepts dey relate.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — MAF Agent wit Cognee Tools

Now we go create one MAF agent we fit query Cognee knowledge graph through `@tool` functions. Dis one go make di agent fit use di full power of graph-aware semantic search plus still keep di conversation context through sessions.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Working Memory wit Sessions

Di `AgentSession` (wey dem dey create wit `agent.create_session()`) dey provide working memory inside session. Di agent fit refer back to earlier messages and still fit query Cognee long-term knowledge graph.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## New Session — Long-Term Memory Dey Persist

When you start fresh session, e go clear working memory, but the Cognee knowledge graph still dey. Di agent fit collect di same long-term knowledge even if na new conversation e be.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

For dis notebook, you don build coding assistant wey combine **MAF's working memory** (`agent.create_session()`) wit **Cognee's long-term knowledge graph**.

### Wetin You Don Learn
1. **How to build knowledge graph**: Cognee sabi manage text wey no get structure den e go build graph + vector memory.
2. **Graph enrichment wit memify**: E fit get derived facts plus better relationships for top your graph wey dey.
3. **MAF + Cognee join body**: `@tool` functions make MAF agent fit query Cognee graph like sey na normal thing.
4. **Working memory + long-term memory**: `AgentSession` (by `agent.create_session()`) dey give session context while Cognee dey give knowledge wey no go forget.
5. **Filtered search wit NodeSets**: You fit target certain parts of knowledge graph (like only principles).

### Main Points
- **Cognee** dey turn raw text to structured, relationship-aware memory — e powerful pass flat vector store.
- **`@tool` functions** connect MAF agents wit knowledge systems outside cleanly.
- **`AgentSession`** (by `agent.create_session()`) dey keep per-conversation context separate from long lasting knowledge.
- Di same knowledge graph fit serve plenty sessions and agents dem.

### How E Fit Work For Real Life
- **Developer copilots**: Code review, incident analysis, architecture assistants
- **Customer-facing copilots**: Support agents wey work wit product docs, FAQs, and CRM notes
- **Internal expert copilots**: Policy, legal, or security assistants wey reason over guidelines
- **Unified data layers**: Join structured and unstructured data inside one graph wey fit get queried

### Next Steps
- Try make Cognee sabi time better
- Define OWL ontology for domain-specific graph quality
- Add user feedback loops to improve retrieval through time
- Scale to multi-agent systems wey dey share di same Cognee memory layer


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
